In [2]:
import pandas as pd

# 1. すでに作成した4番タイプ表を読み込む
# 例: 前段階で保存したファイル名
type_df = pd.read_csv("2025_成績_タイプ.csv")

# 2. NPB公式の2025年チーム打撃成績URL
CENTRAL_URL = "https://npb.jp/bis/2025/stats/tmb_c.html"
PACIFIC_URL = "https://npb.jp/bis/2025/stats/tmb_p.html"

def load_team_runs(url: str) -> pd.DataFrame:
    tables = pd.read_html(url)

    target = None
    for df in tables:
        cols = [str(c).strip() for c in df.columns]
        # 「チーム」「試合」「得点」を含む表を探す
        if "チーム" in cols and "試合" in cols and "得点" in cols:
            target = df.copy()
            break

    if target is None:
        raise ValueError(f"必要な表が見つかりません: {url}")

    # 必要列だけ残す
    target = target[["チーム", "試合", "得点"]].copy()
    return target

# 3. セ・パを読み込んで結合
central_df = load_team_runs(CENTRAL_URL)
pacific_df = load_team_runs(PACIFIC_URL)
team_runs_df = pd.concat([central_df, pacific_df], ignore_index=True)

# 4. 球団名をあなたのCSV側に合わせる
name_map = {
    "ＤｅＮＡ": "DeNA",
    "ディー・エヌ・エー": "DeNA",
    "ソフトバンク": "ソフトバンク",
    "日本ハム": "日本ハム",
    "オリックス": "オリックス",
    "ロッテ": "ロッテ",
    "楽天": "楽天",
    "西武": "西武",
    "巨人": "巨人",
    "阪神": "阪神",
    "広島": "広島",
    "ヤクルト": "ヤクルト",
    "中日": "中日",
}

team_runs_df["球団"] = team_runs_df["チーム"].replace(name_map)

# 5. 数値化
team_runs_df["試合"] = pd.to_numeric(team_runs_df["試合"], errors="coerce")
team_runs_df["得点"] = pd.to_numeric(team_runs_df["得点"], errors="coerce")

# 6. 1試合平均得点を作成
team_runs_df["1試合平均得点"] = team_runs_df["得点"] / team_runs_df["試合"]

# 7. 4番タイプ表と結合
merged_df = type_df.merge(
    team_runs_df[["球団", "試合", "得点", "1試合平均得点"]],
    on="球団",
    how="left"
)

# 8. 保存
merged_df.to_csv("npb_2025_main_fourth_batter_types_with_team_runs.csv",
                 index=False, encoding="utf-8-sig")

print("=== 4番タイプ × チーム得点 ===")
print(merged_df)

print("\n=== タイプ別平均 ===")
summary = (
    merged_df.groupby("4番タイプ")[["得点", "1試合平均得点"]]
    .mean()
    .round(3)
    .reset_index()
)
print(summary)

print("\n保存完了: npb_2025_main_fourth_batter_types_with_team_runs.csv")

=== 4番タイプ × チーム得点 ===
        球団     選手名  4番出場回数  本塁打    OBP    SLG    OPS   打点  長打型スコア  総合型スコア  \
0     DeNA    牧 秀悟      60   16  0.325  0.475  0.800   49   0.042   0.051   
1    オリックス  杉本 裕太郎      74   16  0.332  0.426  0.758   53  -0.221  -0.179   
2   ソフトバンク   山川 穂高      68   23  0.300  0.402  0.702   62   0.024  -0.738   
3     ヤクルト     オスナ      64   14  0.307  0.377  0.684   67  -0.665  -0.812   
4      ロッテ   山本 大斗      47   11  0.262  0.338  0.600   33  -1.150  -1.633   
5       中日   細川 成也      75   20  0.367  0.489  0.856   58   0.443   0.672   
6       巨人   岡本 和真      66   15  0.416  0.598  1.014   49   0.830   1.999   
7       広島   末包 昇大      63   11  0.296  0.373  0.669   62  -0.894  -0.977   
8     日本ハム   野村 佑希      52    8  0.325  0.398  0.723   35  -0.890  -0.449   
9       楽天     ボイト      35   13  0.384  0.498  0.882   39   0.087   0.945   
10      西武     ネビン     119   21  0.346  0.448  0.794   63   0.238   0.141   
11      阪神   佐藤 輝明     126   40  0.345  0.579  0.924  